# M3: Minimal HTTP Server

Put the M2 `Engine` behind `POST /generate`.

```text
client --HTTP JSON--> FastAPI handler --> Engine.generate --> Req --> JSON response
```

The code lives in `server.py`. This notebook starts the real server and sends real HTTP requests to it, to see each status code a client can get:

- `200`: success
- `4xx`: the client sent something wrong
- `5xx`: the server could not serve the request

## Setup

Start uvicorn in a background thread so the notebook can act as the client. It is the same server `python server.py` starts, just in-process.

Port 30000 is SGLang's default. Stop any `python server.py` you started in a terminal first, or the port is taken.

In [1]:
import json
import threading
import time

import httpx
import uvicorn

from engine import Engine
from server import create_app

engine = Engine()
print(engine.device)

server = uvicorn.Server(uvicorn.Config(create_app(engine), host="127.0.0.1", port=30000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
while not server.started:
    time.sleep(0.1)

client = httpx.Client(base_url="http://127.0.0.1:30000", timeout=120)


def show(resp):
    print(resp.status_code, resp.reason_phrase)
    try:
        print(json.dumps(resp.json(), indent=2, ensure_ascii=False))
    except json.JSONDecodeError:
        print(repr(resp.text))  # not every error body is JSON

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

mps


## 200: Success

The body has three layers:

- `text`: what the model said, for humans. Prompt not included.
- `output_ids`: the exact tokens the model produced, for programs.
- `meta_info`: what happened to the request: its `id`, why it finished, token counts.

In [2]:
show(client.post("/generate", json={"text": "The capital of France is", "sampling_params": {"max_new_tokens": 16}}))

200 OK
{
  "text": " Paris. If the number of tourists in Paris is 18, then the",
  "output_ids": [
    12095,
    13,
    1416,
    279,
    1372,
    315,
    31653,
    304,
    12095,
    374,
    220,
    16,
    23,
    11,
    1221,
    279
  ],
  "meta_info": {
    "id": "863bea9c622242a4b2d9301174df86a3",
    "finish_reason": "length",
    "prompt_tokens": 5,
    "completion_tokens": 16
  }
}


In [3]:
# A small limit cuts the model off: finish_reason is "length", completion_tokens equals the limit
show(client.post("/generate", json={"text": "The capital of France is", "sampling_params": {"max_new_tokens": 3}}))

200 OK
{
  "text": " Paris. What",
  "output_ids": [
    12095,
    13,
    3555
  ],
  "meta_info": {
    "id": "405fac0225ca430089573287ae4ee8d7",
    "finish_reason": "length",
    "prompt_tokens": 5,
    "completion_tokens": 3
  }
}


In [4]:
# sampling_params is optional; max_new_tokens defaults to 32
resp = client.post("/generate", json={"text": "Hello"})
print(resp.status_code, resp.json()["meta_info"])

200 {'id': '923a075171164e1e833eafdc72652e03', 'finish_reason': 'length', 'prompt_tokens': 1, 'completion_tokens': 32}


## 4xx: The Client Sent Something Wrong

Retrying the same request will fail again. The client has to change the request.

- `422 Unprocessable Entity`: the JSON does not match `GenerateReqInput`. FastAPI and pydantic reject it before the handler runs, and `detail` says which field is wrong.
- `400 Bad Request`: the JSON has the right shape, but the value is invalid. The handler maps the Engine's `ValueError` to 400.
- `404` and `405`: FastAPI routing. The path does not exist, or the path exists but not for this HTTP method.

In [5]:
# 422: required field `text` is missing
show(client.post("/generate", json={"sampling_params": {"max_new_tokens": 5}}))

422 Unprocessable Entity
{
  "detail": [
    {
      "type": "missing",
      "loc": [
        "body",
        "text"
      ],
      "msg": "Field required",
      "input": {
        "sampling_params": {
          "max_new_tokens": 5
        }
      }
    }
  ]
}


In [6]:
# 422: wrong type, max_new_tokens is not an int
show(client.post("/generate", json={"text": "Hello", "sampling_params": {"max_new_tokens": "many"}}))

422 Unprocessable Entity
{
  "detail": [
    {
      "type": "int_parsing",
      "loc": [
        "body",
        "sampling_params",
        "max_new_tokens"
      ],
      "msg": "Input should be a valid integer, unable to parse string as an integer",
      "input": "many"
    }
  ]
}


In [7]:
# 400: right type, invalid value. Pydantic accepts 0; the Engine rejects it.
show(client.post("/generate", json={"text": "Hello", "sampling_params": {"max_new_tokens": 0}}))

400 Bad Request
{
  "detail": "max_new_tokens must be >= 1, got 0"
}


In [8]:
# 404: no such path
show(client.post("/v1/completions", json={"prompt": "Hello"}))

404 Not Found
{
  "detail": "Not Found"
}


In [9]:
# 405: /generate exists, but only for POST
show(client.get("/generate"))

405 Method Not Allowed
{
  "detail": "Method Not Allowed"
}


## 5xx: The Server Could Not Serve the Request

The request itself is fine. The failure is on the server side.

### 503: Busy

The engine has capacity for one request. Send a long request from a background thread, then send a second request while it runs. The second request gets `503 Service Unavailable` right away instead of waiting.

A 503 means "try again later": the same request can succeed once the engine is idle. M4 replaces this rejection with a queue.

In [10]:
results = {}

long_client = threading.Thread(
    target=lambda: results.setdefault(
        "A", client.post("/generate", json={"text": "Write a long story about a dragon.", "sampling_params": {"max_new_tokens": 200}})
    )
)
long_client.start()
while engine.running_req is None:  # wait until A is admitted
    time.sleep(0.01)

start = time.perf_counter()
show(client.post("/generate", json={"text": "Hello"}))  # B arrives while A is running
print(f"B answered in {time.perf_counter() - start:.3f}s")

long_client.join()
print("A:", results["A"].status_code, results["A"].json()["meta_info"])

503 Service Unavailable
{
  "detail": "engine is busy with another request"
}
B answered in 0.003s
A: 200 {'id': '394c281c8a2f452abd570c75f5afbddf', 'finish_reason': 'length', 'prompt_tokens': 8, 'completion_tokens': 200}


## Summary

| Status | Who is wrong | Retry the same request? | Where it comes from |
|---|---|---|---|
| 200 | nobody | - | handler returns `GenerateResponse` |
| 400 | client | no, fix the value | handler catches `ValueError` |
| 404 / 405 | client | no, fix the path or method | FastAPI routing |
| 422 | client | no, fix the JSON shape | pydantic, before the handler runs |
| 503 | nobody, the server is full | yes, later | handler catches `EngineBusyError` |

## Shutdown

In [11]:
client.close()
server.should_exit = True